# 01 数据准备与质量审核

**目标：** 核对 PlantVillage 与 PlantDoc 的现有数据，确定共同类别，完成样本审核与固定划分。

**前置条件：** 已完成数据目录整理。本 Notebook 的统计与审核展示只需基础依赖；正式划分需要先确定标签、样本取舍和分组。

**输入：** `data/metadata/` 中的已有清单、`configs/labels.yaml`、原始图片。

**输出：** 经审核后生成 `data/splits/pv_train.csv`、`pv_val.csv`、`pv_test.csv` 和 `plantdoc_external_test.csv`。默认只查看已有证据，不提前生成实验划分。

请在项目根目录或 `notebooks/` 中启动 Jupyter；每本 Notebook 都能独立从头运行。

In [2]:
from pathlib import Path
from copy import deepcopy
import csv
import html
import json
import sys


def find_notebook_project_root():
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        for root in (candidate, candidate / "plant-disease-project"):
            if (root / "configs" / "base.yaml").is_file():
                return root
    raise FileNotFoundError("请从项目根目录或其 notebooks 目录启动 Jupyter。")


PROJECT_ROOT = find_notebook_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils import load_config

BASE = load_config(PROJECT_ROOT / "configs" / "base.yaml")
EXPERIMENTS = load_config(PROJECT_ROOT / "configs" / "experiments.yaml")
LABELS = load_config(PROJECT_ROOT / "configs" / "labels.yaml")


def read_csv(path):
    path = Path(path)
    if not path.exists():
        return []
    with path.open(encoding="utf-8-sig", newline="") as handle:
        return list(csv.DictReader(handle))


def show_table(records, columns=None, limit=20):
    records = list(records)
    if not records:
        print("暂无记录。")
        return
    columns = columns or list(records[0])
    rows = records[:limit]
    header = "".join(f"<th>{html.escape(str(c))}</th>" for c in columns)
    body = "".join("<tr>" + "".join(
        f"<td>{html.escape(str(row.get(c, '')))}</td>" for c in columns
    ) + "</tr>" for row in rows)
    try:
        from IPython.display import HTML, display
        display(HTML(f"<table><thead><tr>{header}</tr></thead><tbody>{body}</tbody></table>"))
    except ImportError:
        print(json.dumps(rows, ensure_ascii=False, indent=2))
    if len(records) > limit:
        print(f"仅展示前 {limit} 行，共 {len(records)} 行。")


print("项目目录：", PROJECT_ROOT)
print("已配置类别数：", len(LABELS.get("classes", [])))

项目目录： /Users/qingjiarui/Desktop/深度学习课程作业/基于迁移学习的植物叶片病害识别与真实场景泛化优化研究/plant-disease-project
已配置类别数： 5


## 1. 查看真实数据清单

这里读取整理阶段保存的统计，不将彩色、灰度和分割版本视为独立生物样本。主实验先使用 PlantVillage 彩色版本；PlantDoc 的来源 `train/test` 目录只记录来源，最终用途以固定外部测试清单为准。

In [3]:
counts_path = PROJECT_ROOT / "data" / "metadata" / "class_counts.csv"
counts = read_csv(counts_path)
print("已有类别统计表：", counts_path)
if counts:
    show_table(counts, limit=15)
    totals = {}
    for row in counts:
        key = (row.get("dataset", ""), row.get("variant", ""), row.get("split", ""))
        totals[key] = totals.get(key, 0) + int(row.get("image_count", 0))
    show_table([dict(dataset=k[0], variant=k[1], source_split=k[2], image_count=v)
                for k, v in sorted(totals.items())])
else:
    print("未找到已有类别统计；请先核对 data/metadata 的整理结果。")

summary_path = PROJECT_ROOT / "data" / "metadata" / "summary.json"
if summary_path.exists():
    summary = load_config(summary_path)
    print("整理摘要：")
    print(json.dumps(summary, ensure_ascii=False, indent=2))

已有类别统计表： /Users/qingjiarui/Desktop/深度学习课程作业/基于迁移学习的植物叶片病害识别与真实场景泛化优化研究/plant-disease-project/data/metadata/class_counts.csv


dataset,variant,split,original_class,class_name,quarantined,image_count
plantdoc,,test,Apple Scab Leaf,apple_scab_leaf,False,10
plantdoc,,test,Apple leaf,apple_leaf,False,9
plantdoc,,test,Apple rust leaf,apple_rust_leaf,False,10
plantdoc,,test,Bell_pepper leaf,bell_pepper_leaf,False,8
plantdoc,,test,Bell_pepper leaf spot,bell_pepper_leaf_spot,False,9
plantdoc,,test,Blueberry leaf,blueberry_leaf,False,11
plantdoc,,test,Cherry leaf,cherry_leaf,False,10
plantdoc,,test,Corn Gray leaf spot,corn_gray_leaf_spot,False,4
plantdoc,,test,Corn leaf blight,corn_leaf_blight,False,12
plantdoc,,test,Corn rust leaf,corn_rust_leaf,False,10


仅展示前 15 行，共 169 行。


dataset,variant,source_split,image_count
plantdoc,,test,236
plantdoc,,train,2336
plantvillage,color,,54305
plantvillage,grayscale,,54305
plantvillage,segmented,,54306


整理摘要：
{
  "status": "complete",
  "organized_on": "2026-10-18",
  "source_file_count": 165494,
  "retained_original_file_count": 165491,
  "retained_original_bytes": 3175627498,
  "deleted_system_cache_files": 3,
  "deleted_bytes": 26636,
  "dataset_images": 165488,
  "documentation_images": 1,
  "quarantined_images": 0,
  "verified_sha256_files": 165491,
  "image_counts": {
    "plantvillage_color": 54305,
    "plantvillage_grayscale": 54305,
    "plantvillage_segmented": 54306,
    "plantdoc_train": 2336,
    "plantdoc_test": 236
  },
  "class_counts": {
    "plantvillage": 38,
    "plantdoc": 28
  },
  "duplicate_groups_preserved": 75,
  "cross_split_duplicate_groups": 11,
  "all_retained_original_content_unchanged": true
}


## 2. 核对命名映射与重复样本

`class_mapping.csv` 记录来源目录的命名对应；正式训练使用 `labels.yaml` 中统一的 `label_index`。`duplicate_review.csv` 是待核查证据，其中的原整理动作不代表已经完成实验审核。

审核决定记录在 `review_decisions.csv`；来源叶片与重复图片的分组记录在 `sample_groups.csv`。对应的彩色、灰度与分割版本编号不能直接证明不同照片来自不同叶片。存在标签冲突时，先完成取舍并记录理由，再划分。

In [4]:
mapping_rows = read_csv(PROJECT_ROOT / "data" / "metadata" / "class_mapping.csv")
duplicate_rows = read_csv(PROJECT_ROOT / "data" / "metadata" / "duplicate_review.csv")
print("来源类别映射预览：")
show_table(mapping_rows, limit=12)
print("重复与冲突审核清单行数：", len(duplicate_rows))
show_table(duplicate_rows,
           columns=["group_id", "issue_type", "dataset", "split", "class_name", "new_path"],
           limit=12)

for filename in ("review_decisions.csv", "sample_groups.csv"):
    path = PROJECT_ROOT / "data" / "metadata" / filename
    rows = read_csv(path)
    print(f"{filename}：{len(rows)} 条记录")
    show_table(rows, limit=5)

来源类别映射预览：


dataset,class_index,original_class,class_name
plantvillage,0,Apple___Apple_scab,apple_apple_scab
plantvillage,1,Apple___Black_rot,apple_black_rot
plantvillage,2,Apple___Cedar_apple_rust,apple_cedar_apple_rust
plantvillage,3,Apple___healthy,apple_healthy
plantvillage,4,Blueberry___healthy,blueberry_healthy
plantvillage,5,Cherry_(including_sour)___healthy,cherry_including_sour_healthy
plantvillage,6,Cherry_(including_sour)___Powdery_mildew,cherry_including_sour_powdery_mildew
plantvillage,7,Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot,corn_maize_cercospora_leaf_spot_gray_leaf_spot
plantvillage,8,Corn_(maize)___Common_rust_,corn_maize_common_rust
plantvillage,9,Corn_(maize)___healthy,corn_maize_healthy


仅展示前 12 行，共 66 行。
重复与冲突审核清单行数： 150


group_id,issue_type,dataset,split,class_name,new_path
1,cross_split_duplicate,plantdoc,test,blueberry_leaf,plantdoc/test/blueberry_leaf/pd_test_blueberry_leaf_000003.jpg
1,cross_split_duplicate,plantdoc,train,blueberry_leaf,plantdoc/train/blueberry_leaf/pd_train_blueberry_leaf_000066.jpg
2,cross_split_label_conflict,plantdoc,test,corn_gray_leaf_spot,plantdoc/test/corn_gray_leaf_spot/pd_test_corn_gray_leaf_spot_000001.jpg
2,cross_split_label_conflict,plantdoc,train,corn_leaf_blight,plantdoc/train/corn_leaf_blight/pd_train_corn_leaf_blight_000066.jpg
3,cross_split_label_conflict,plantdoc,test,corn_leaf_blight,plantdoc/test/corn_leaf_blight/pd_test_corn_leaf_blight_000008.jpg
3,cross_split_label_conflict,plantdoc,train,corn_gray_leaf_spot,plantdoc/train/corn_gray_leaf_spot/pd_train_corn_gray_leaf_spot_000008.jpg
4,cross_split_label_conflict,plantdoc,test,corn_leaf_blight,plantdoc/test/corn_leaf_blight/pd_test_corn_leaf_blight_000012.jpg
4,cross_split_label_conflict,plantdoc,train,corn_gray_leaf_spot,plantdoc/train/corn_gray_leaf_spot/pd_train_corn_gray_leaf_spot_000041.jpg
5,cross_split_label_conflict,plantdoc,test,potato_leaf_early_blight,plantdoc/test/potato_leaf_early_blight/pd_test_potato_leaf_early_blight_000002.jpg
5,cross_split_label_conflict,plantdoc,train,potato_leaf_late_blight,plantdoc/train/potato_leaf_late_blight/pd_train_potato_leaf_late_blight_000049.jpg


仅展示前 12 行，共 150 行。
review_decisions.csv：8310 条记录


image_path,decision,reason
data/plantdoc/test/tomato_early_blight_leaf/pd_test_tomato_early_blight_leaf_000001.jpg,exclude,已核对图像：幻灯片/文档/插图或多图拼版，非单一自然场景番茄叶片照片，保守排除；视觉内容筛查：内容筛查排除：带说明文字的教学幻灯片，含两幅独立照片拼接，不是单场景叶片照片
data/plantdoc/test/tomato_early_blight_leaf/pd_test_tomato_early_blight_leaf_000002.jpg,keep,沿用PlantDoc原数据集类别；已完成图像解码、哈希/重复检查与整类缩略图初筛，未见明确排除依据；非专家病理确诊
data/plantdoc/test/tomato_early_blight_leaf/pd_test_tomato_early_blight_leaf_000003.jpg,keep,沿用PlantDoc原数据集类别；已完成图像解码、哈希/重复检查与整类缩略图初筛，未见明确排除依据；非专家病理确诊
data/plantdoc/test/tomato_early_blight_leaf/pd_test_tomato_early_blight_leaf_000004.jpg,keep,沿用PlantDoc原数据集类别；已完成图像解码、哈希/重复检查与整类缩略图初筛，未见明确排除依据；非专家病理确诊
data/plantdoc/test/tomato_early_blight_leaf/pd_test_tomato_early_blight_leaf_000005.jpg,keep,沿用PlantDoc原数据集类别；已完成图像解码、哈希/重复检查与整类缩略图初筛，未见明确排除依据；非专家病理确诊


仅展示前 5 行，共 8310 行。
sample_groups.csv：6439 条记录


image_path,group_id,group_method,reviewed
data/plantdoc/test/tomato_early_blight_leaf/pd_test_tomato_early_blight_leaf_000002.jpg,pd_tomato_early_blight_leaf_provenance_uncertain_unsplit,保守外测来源桶：原文件名可追溯但生物叶片/采集来源身份未知；核对hash与相似图，同类全体保留图仅用于external_test且不拆分；此桶不表示单片叶片身份已确认,true
data/plantdoc/test/tomato_early_blight_leaf/pd_test_tomato_early_blight_leaf_000003.jpg,pd_tomato_early_blight_leaf_provenance_uncertain_unsplit,保守外测来源桶：原文件名可追溯但生物叶片/采集来源身份未知；核对hash与相似图，同类全体保留图仅用于external_test且不拆分；此桶不表示单片叶片身份已确认,true
data/plantdoc/test/tomato_early_blight_leaf/pd_test_tomato_early_blight_leaf_000004.jpg,pd_tomato_early_blight_leaf_provenance_uncertain_unsplit,保守外测来源桶：原文件名可追溯但生物叶片/采集来源身份未知；核对hash与相似图，同类全体保留图仅用于external_test且不拆分；此桶不表示单片叶片身份已确认,true
data/plantdoc/test/tomato_early_blight_leaf/pd_test_tomato_early_blight_leaf_000005.jpg,pd_tomato_early_blight_leaf_provenance_uncertain_unsplit,保守外测来源桶：原文件名可追溯但生物叶片/采集来源身份未知；核对hash与相似图，同类全体保留图仅用于external_test且不拆分；此桶不表示单片叶片身份已确认,true
data/plantdoc/test/tomato_early_blight_leaf/pd_test_tomato_early_blight_leaf_000006.jpg,pd_tomato_early_blight_leaf_provenance_uncertain_unsplit,保守外测来源桶：原文件名可追溯但生物叶片/采集来源身份未知；核对hash与相似图，同类全体保留图仅用于external_test且不拆分；此桶不表示单片叶片身份已确认,true


仅展示前 5 行，共 6439 行。


## 3. 确定共同类别

先从研究方案中的候选共同类别筛选约 4～6 类，并核对两数据集的病害语义与来源目录。不要仅凭相似目录名自动认定类别对应。

请编辑 `configs/labels.yaml`：每类包含 `label`、从 0 连续编号的 `label_index`、`plantvillage_class`、`plantdoc_classes`，可选 `display_name`。同时按配置文件中的审核字段记录审核状态。

In [5]:
classes = LABELS.get("classes", [])
if classes:
    show_table(classes)
    print("请核对类别语义、编号连续性及每一来源目录，再生成正式划分。")
else:
    print("尚未确定最终共同类别：请先编辑 configs/labels.yaml 的 classes。")
    print("目前不生成训练、验证、测试清单。")

print("主实验使用 PlantVillage color；默认目标比例为 70% / 15% / 15%。")
print("先按来源叶片/重复组分组，再尽量保持类别比例；PlantDoc 全部保留作外部评价。")

label,label_index,plantvillage_class,plantdoc_classes,display_name
tomato_bacterial_spot,0,tomato_bacterial_spot,['tomato_leaf_bacterial_spot'],番茄细菌斑病
tomato_early_blight,1,tomato_early_blight,['tomato_early_blight_leaf'],番茄早疫病
tomato_late_blight,2,tomato_late_blight,['tomato_leaf_late_blight'],番茄晚疫病
tomato_leaf_mold,3,tomato_leaf_mold,['tomato_mold_leaf'],番茄叶霉病
tomato_septoria_leaf_spot,4,tomato_septoria_leaf_spot,['tomato_septoria_leaf_spot'],番茄斑枯病


请核对类别语义、编号连续性及每一来源目录，再生成正式划分。
主实验使用 PlantVillage color；默认目标比例为 70% / 15% / 15%。
先按来源叶片/重复组分组，再尽量保持类别比例；PlantDoc 全部保留作外部评价。


## 4. 审核完成后生成固定清单

将下面的 `PREPARE_SPLITS` 改为 `True` 才会调用准备流程。流程应验证配置、审核决定和分组信息，发现缺失时会停止并指出原因。

第一次生成后，后续实验统一读取这些清单。模型比较过程中不要反复更改划分，也不要利用 PlantDoc 指标选择模型、增强方式或训练轮次。

In [6]:
PREPARE_SPLITS = False

if not PREPARE_SPLITS:
    print("当前为审核模式。完成标签和分组审核后，将 PREPARE_SPLITS 设为 True。")
elif not LABELS.get("classes"):
    print("未配置共同类别，暂不能生成正式划分。")
else:
    from src.data import prepare_manifests
    result = prepare_manifests(
        PROJECT_ROOT,
        LABELS,
        seed=BASE.get("seed", 42),
        ratios=(0.7, 0.15, 0.15),
    )
    print(json.dumps(result, ensure_ascii=False, indent=2, default=str))

split_counts = []
for split_name in ("pv_train", "pv_val", "pv_test", "plantdoc_external_test"):
    path = PROJECT_ROOT / "data" / "splits" / f"{split_name}.csv"
    split_counts.append({"split": split_name, "exists": path.exists(), "samples": len(read_csv(path))})
show_table(split_counts)

当前为审核模式。完成标签和分组审核后，将 PREPARE_SPLITS 设为 True。


split,exists,samples
pv_train,True,4194
pv_val,True,908
pv_test,True,896
plantdoc_external_test,True,441


## 结论记录


**1. 共同类别与选择依据**

选取番茄细菌斑病、早疫病、晚疫病、叶霉病和斑枯病，共5类，统一标签编号为0—4。选择依据为：两数据集中存在对应类别，审核后各类仍有足够样本，且保留的PlantVillage图片能匹配官方叶片分组。限定同一作物，有助于集中研究病害分类与跨场景泛化。

**2. 标签冲突、重复样本的处置及理由**

共审核8,310张候选图片，保留6,439张、排除1,871张。其中，PlantVillage因缺少可靠官方分组依据排除1,761张，保留5,998张；PlantDoc因重复、标签冲突、明确的类别存疑及教学幻灯片等问题排除110张，保留441张。

同一照片的重复、缩放、镜像或裁切版本仅保留一个合格代表；存在跨类别标签冲突的相关版本全部排除，不自行猜测或修改病害标签。所有排除均记录理由，原始图片保留。

**3. 来源叶片分组证据与无法确认的限制**

PlantVillage依据原始文件名及类别命名空间，精确匹配官方叶片分组表，共获得1,500个来源组，同一组不跨训练、验证和测试集合。

PlantDoc无法可靠恢复生物叶片身份，因此按类别建立5个保守分组，全部固定用于外部测试，不参与训练、调参和早停。这5组不代表只有5片叶子；`reviewed=true`表示分组记录与使用政策已核对。

病害标签沿用源数据标注，未进行专家病理学确诊；近重复筛查也不能保证发现所有同源图片。研究结论应限定于本次固定数据子集。

**4. 固定清单生成日期、各集合类别分布**

生成日期为 **2026年10月20日**，随机种子为 **42**。PlantVillage按来源组以约70%／15%／15%划分，实际图片数量如下：

| 类别 | 训练集 | 验证集 | 同域测试集 | PlantDoc外部测试集 |
|---|---:|---:|---:|---:|
| 番茄细菌斑病 | 1,487 | 320 | 320 | 73 |
| 番茄早疫病 | 699 | 152 | 148 | 76 |
| 番茄晚疫病 | 644 | 140 | 136 | 102 |
| 番茄叶霉病 | 664 | 144 | 144 | 73 |
| 番茄斑枯病 | 700 | 152 | 148 | 117 |
| **合计** | **4,194** | **908** | **896** | **441** |

**5. 本次审核是否允许开始E1**

**允许开始E1。** 类别配置已设为`approved`，审核记录、来源分组和四份正式清单均已生成，并通过候选覆盖、标签一致性、跨划分重复与来源组检查，以及数据加载验证。E1仅使用PlantVillage训练集和验证集开展模型比较，同域测试集与PlantDoc外部测试集保留至最终评价阶段使用。